# Çatı GES — SAM segmentasyonu, BİRLEŞİK (Atatürk OSB + Çiğli kamu, 1085 çatı)
Girdi: `CatiGunes_SAM_Birlesik.zip` (tek dosya: görüntüler, ayak izi maskeleri, `sam_girdi.csv`, harita bilgisi, önceki sonuçlar).
Ayak izleri 2026-10-22 elle düzeltilmiş haliyle (24 çatı, `hizalama = elle_duzeltildi`).

**Çalışma zamanı → T4 GPU.** Sıra: 1 → 2 (zip seç) → 3 → 4 → 5 → 6 (indir) → 7 (harita indir) → 8 (poligon indir). Multimask karşılaştırması: 1 → 2 → 3 → 9 → 10 → 11.
4. hücrede `YENIDEN_HEPSI = False` (varsayılan): önceki sonuçlar alınır, yalnız eksik çatılar koşulur (birkaç dk).
`True`: 1085 çatının hepsi baştan (~40 dk; kesilirse 4'ü tekrar çalıştır, kaldığı yerden devam eder). **Defteri tek sekmede aç.**

In [2]:
# 1) GPU + SAM kurulumu ve model indirme
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install git+https://github.com/facebookresearch/segment-anything.git folium rasterio
# model her zaman /content'e iner (-c: yarım kaldıysa tamamlar, tam ise atlar)
!wget -q -c -P /content https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth
!ls -la /content/*.pth
print("kurulum ve model indirme tamam")

name, memory.total [MiB]
Tesla T4, 15360 MiB
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
-rw-r--r-- 1 root root 2564550879 Apr  4  2023 /content/sam_vit_h_4b8939.pth
kurulum ve model indirme tamam


In [3]:
# 2) Veriyi yükle: CatiGunes_SAM_Birlesik.zip (zaten açıldıysa atlar)
from google.colab import files
import zipfile, os
os.chdir("/content")
if not os.path.exists("/content/veri/sam_girdi.csv"):
    yuklenen = files.upload()
    for ad in yuklenen:
        zipfile.ZipFile(ad).extractall("/content/veri")
os.chdir("/content/veri")
assert os.path.exists("sam_girdi.csv"), "yanlış zip: CatiGunes_SAM_Birlesik.zip seçilmeli"
print(sorted(f for f in os.listdir(".") if not f.startswith(("goruntuler", "maskeler"))))

Saving CatiGunes_SAM_Birlesik.zip to CatiGunes_SAM_Birlesik.zip
['BENIOKU.md', 'georef.csv', 'onceki', 'panelsiz', 'sam_birlesik_colab.ipynb', 'sam_girdi.csv', 'sinirlar.geojson']


In [4]:
# 3) Modeli yükle
import torch
from segment_anything import sam_model_registry, SamPredictor
cihaz = "cuda" if torch.cuda.is_available() else "cpu"
sam = sam_model_registry["vit_h"](checkpoint="/content/sam_vit_h_4b8939.pth").to(cihaz)
predictor = SamPredictor(sam)
print("model hazır:", cihaz)

model hazır: cuda


In [5]:
# 4) Kutu istemiyle segmentasyon — OSB + kamu (sam_girdi.csv)
import os, ast, time, shutil, numpy as np, pandas as pd
from PIL import Image
os.chdir("/content/veri")
YENIDEN_HEPSI = True   # True: hepsi baştan; False: onceki/ sonuçları alınır, yalnız eksikler koşulur
df = pd.read_csv("sam_girdi.csv")
assert len(df) == 1085, f"1085 çatı bekleniyordu, {len(df)} geldi"
CIKTI = "sam_sonuc.csv"
os.makedirs("sam_maskeler", exist_ok=True)
if YENIDEN_HEPSI and os.path.exists(CIKTI) and not os.path.exists(".yeniden_basladi"):
    os.remove(CIKTI)
if YENIDEN_HEPSI: open(".yeniden_basladi", "w").close()
if not YENIDEN_HEPSI and not os.path.exists(CIKTI):
    shutil.copy("onceki/sam_sonuc.csv", CIKTI)
    for f in os.listdir("onceki/sam_maskeler"): shutil.copy(os.path.join("onceki/sam_maskeler", f), "sam_maskeler")
    print("önceki sonuçlar alındı")
bitmis = set(pd.read_csv(CIKTI).cati_id) if os.path.exists(CIKTI) else set()
kalan = df[~df.cati_id.isin(bitmis)]
print(f"{len(df)} çatı, {len(bitmis & set(df.cati_id))} bitmiş, {len(kalan)} kalan")
sonuc, t0, BUGUN = [], time.time(), time.strftime("colab_%Y-%m-%d")
def yaz():
    global sonuc
    if sonuc:
        pd.DataFrame(sonuc).to_csv(CIKTI, mode="a", header=not os.path.exists(CIKTI), index=False)
        sonuc = []
for k, r in enumerate(kalan.itertuples(), 1):
    img = np.array(Image.open(r.goruntu).convert("RGB"))
    kutu = np.array(ast.literal_eval(r.kutu_px), dtype=float)
    ayak = np.array(Image.open(r.maske)) > 0
    predictor.set_image(img)
    masks, scores, _ = predictor.predict(box=kutu, multimask_output=True)
    iou = [(mm & ayak).sum() / max((mm | ayak).sum(), 1) for mm in masks]
    i = int(np.argmax(np.array(iou) + 1e-3 * scores))
    ham = masks[i]; m = ham & ayak
    px2 = r.yer_m_px ** 2
    Image.fromarray((m * 255).astype(np.uint8)).save(f"sam_maskeler/c{r.cati_id:04d}.png")
    sonuc.append(dict(cati_id=r.cati_id, maske_dosyasi=f"sam_maskeler/c{r.cati_id:04d}.png",
        sam_alan_m2=round(m.sum() * px2, 1), ayak_izi_m2=round(ayak.sum() * px2, 1),
        kaplama=round(m.sum() / max(ayak.sum(), 1), 3), tasma=round((ham & ~ayak).sum() / max(ham.sum(), 1), 3),
        iou_ayak=round(float(iou[i]), 3), guven=round(float(scores[i]), 3), secilen_maske=i, kosu=BUGUN))
    if k % 50 == 0:
        yaz(); h = (time.time() - t0) / k
        print(f"{k}/{len(kalan)}  ~{h:.1f} sn/çatı, kalan ~{h*(len(kalan)-k)/60:.0f} dk")
yaz()
if os.path.exists(".yeniden_basladi"): os.remove(".yeniden_basladi")
res = pd.read_csv(CIKTI).drop_duplicates("cati_id", keep="last")
res = res[res.cati_id.isin(df.cati_id)]
res.to_csv(CIKTI, index=False)
assert set(res.cati_id) == set(df.cati_id), "sonuç dosyasında eksik çatı var"
# şüpheli kuralı (tüm koşularda aynı): kaplama < 0,7 | taşma > 0,3 | güven < 0,85 | şüpheli bölüm | görsel sorun
s = df.drop(columns=["goruntu", "maske"]).merge(res, on="cati_id")
dk = s.dislama_nedeni.fillna("").astype(str)
s["supheli"] = ((s.kaplama < 0.7) | (s.tasma > 0.3) | (s.guven < 0.85) | (s.bolum == "supheli") | dk.str.contains("gorsel_sorun"))
s.to_csv("sam_sonuc_tam.csv", index=False)
print("bitti:", len(s), "çatı | şüpheli:", int(s.supheli.sum()), "| koşu:", s.kosu.value_counts().to_dict())
print("SAM toplam %.0f m² / ayak izi %.0f m² = %%%.1f" % (s.sam_alan_m2.sum(), s.ayak_izi_m2.sum(), 100*s.sam_alan_m2.sum()/s.ayak_izi_m2.sum()))
s.groupby(["bolge", "tur"]).agg(cati=("cati_id", "size"), kaplama_medyan=("kaplama", "median"),
    sam_m2=("sam_alan_m2", "sum"), supheli=("supheli", "sum")).round(3)

1085 çatı, 0 bitmiş, 1085 kalan
50/1085  ~1.9 sn/çatı, kalan ~33 dk
100/1085  ~2.0 sn/çatı, kalan ~33 dk
150/1085  ~2.1 sn/çatı, kalan ~32 dk
200/1085  ~2.1 sn/çatı, kalan ~31 dk
250/1085  ~2.1 sn/çatı, kalan ~29 dk
300/1085  ~2.1 sn/çatı, kalan ~28 dk
350/1085  ~2.1 sn/çatı, kalan ~26 dk
400/1085  ~2.1 sn/çatı, kalan ~24 dk
450/1085  ~2.2 sn/çatı, kalan ~23 dk
500/1085  ~2.2 sn/çatı, kalan ~21 dk
550/1085  ~2.2 sn/çatı, kalan ~19 dk
600/1085  ~2.2 sn/çatı, kalan ~17 dk
650/1085  ~2.2 sn/çatı, kalan ~16 dk
700/1085  ~2.2 sn/çatı, kalan ~14 dk
750/1085  ~2.2 sn/çatı, kalan ~12 dk
800/1085  ~2.2 sn/çatı, kalan ~10 dk
850/1085  ~2.2 sn/çatı, kalan ~9 dk
900/1085  ~2.2 sn/çatı, kalan ~7 dk
950/1085  ~2.2 sn/çatı, kalan ~5 dk
1000/1085  ~2.2 sn/çatı, kalan ~3 dk
1050/1085  ~2.2 sn/çatı, kalan ~1 dk
bitti: 1085 çatı | şüpheli: 128 | koşu: {'colab_2026-10-30': 1085}
SAM toplam 2214231 m² / ayak izi 2433476 m² = %91.0


cati  kaplama_medyan     sam_m2  supheli
bolge       tur                                                  
ataturk_osb sanayi        997           0.922  2051349.3      106
            sanayi_disi     8           0.958    12105.7        1
cigli_kamu  hastane         3           0.898    22042.6        1
            okul           48           0.847    32517.8       18
            universite     29           0.938    96215.4        2

In [6]:
# 5) Görsel kontrol: en düşük kaplamalı 20 çatı (kırmızı kutu, sarı ayak izi, mavi SAM)
import matplotlib.pyplot as plt
tablo = pd.read_csv("sam_girdi.csv").set_index("cati_id")
goster = s.nsmallest(20, "kaplama")
sut = 5; sat = (len(goster) + sut - 1) // sut
fig, ax = plt.subplots(sat, sut, figsize=(4 * sut, 4 * sat))
for a, q in zip(ax.flat, goster.itertuples()):
    r = tablo.loc[q.cati_id]
    img = np.array(Image.open(r.goruntu).convert("RGB"))
    m = np.array(Image.open(q.maske_dosyasi)) > 0
    a.imshow(img); a.imshow(np.ma.masked_where(~m, m), alpha=0.45, cmap="winter")
    x1, y1, x2, y2 = ast.literal_eval(r.kutu_px)
    a.add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, ec="red", lw=1.5))
    p = np.array(ast.literal_eval(r.poligon_px)); a.plot(p[:, 0], p[:, 1], c="yellow", lw=1)
    a.set_title(f"c{q.cati_id:04d} {q.bolge[:5]} {q.tur} {int(q.ayak_izi_m2)}m² kap={q.kaplama}", fontsize=8,
                color="red" if q.supheli else "black")
    a.axis("off")
for a in list(ax.flat)[len(goster):]: a.axis("off")
plt.tight_layout(); plt.savefig("sam_onizleme.jpg", dpi=110); plt.show()

Output hidden; open in https://colab.research.google.com to view.

In [7]:
# 6) Sonuçları indir: sam_sonuc_tam.csv + maskeler + önizleme
import os, zipfile
from google.colab import files
os.chdir("/content/veri")
with zipfile.ZipFile("/content/sam_sonuc_birlesik.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in ["sam_sonuc_tam.csv", "sam_onizleme.jpg"]:
        if os.path.exists(f): z.write(f)
    for f in sorted(os.listdir("sam_maskeler")):
        z.write(os.path.join("sam_maskeler", f))
files.download("/content/sam_sonuc_birlesik.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [8]:
# 7) Harita (OSB + kamu): ham → hizalı ayak izi, kayma okları, oturtma güveni, SAM maskesi
import os, numpy as np, pandas as pd, geopandas as gpd, folium
from PIL import Image
from google.colab import files
from shapely import wkt
from shapely.ops import unary_union
from shapely.geometry import shape, LineString
from rasterio.features import shapes
from affine import Affine
os.chdir("/content/veri")
g = pd.read_csv("georef.csv")
s = pd.read_csv("sam_sonuc_tam.csv")
d = s.merge(g.drop(columns=["bolge", "tur", "alan_m2", "ayak_izi_kaynagi", "gorsel_kontrol"]), on="cati_id", how="left")
assert d.jgw_a.notna().all(), "georef'te eşi olmayan çatı var"
d["kayma_m"] = np.hypot(d.kayma_dogu_m, d.kayma_kuzey_m).round(2)
d["oturtma"] = np.select([d.hizalama == "elle_duzeltildi", d.hizalama_kazanci >= 1.5, d.hizalama_kazanci >= 1.15],
                         ["elle düzeltildi", "yüksek", "orta"], "düşük (yalnız blok)")
d["supheli"] = d["supheli"].map({True: "EVET", False: "hayır"})
for c in ["dislama_nedeni", "mevcut_ges", "kurum_ad"]: d[c] = d[c].fillna("").astype(str)
geo = []
for r in d.itertuples():
    m = np.array(Image.open(r.maske_dosyasi)) > 0
    aff = Affine(r.jgw_a, 0, r.jgw_c - r.jgw_a / 2, 0, r.jgw_e, r.jgw_f - r.jgw_e / 2)
    p = [shape(gm) for gm, v in shapes(m.astype(np.uint8), mask=m, transform=aff) if v == 1]
    geo.append(unary_union(p).simplify(0.3) if p else None)
alanlar = ["cati_id", "bolge", "tur", "kurum_ad", "alan_sinifi", "alan_m2", "sam_alan_m2", "kaplama", "iou_ayak", "guven",
           "supheli", "dislama_nedeni", "oturtma", "hizalama_kazanci", "kayma_m", "ayak_izi_kaynagi", "uydu", "mevcut_ges"]
etiket = ["çatı", "bölge", "tür", "kurum", "alan sınıfı", "ayak izi m²", "SAM m²", "kaplama", "IoU", "SAM skoru",
          "şüpheli", "dikkat", "oturtma", "hizalama kazancı", "kayma m", "ayak izi kaynağı", "uydu", "mevcut GES"]
sam_g = gpd.GeoDataFrame(d[alanlar], geometry=geo, crs=3857).dropna(subset=["geometry"]).to_crs(4326)
hizali = gpd.GeoDataFrame(d[alanlar], geometry=d.wkt_hizali.map(wkt.loads), crs=4326)
ham = gpd.GeoDataFrame(d[["cati_id", "ayak_izi_kaynagi"]], geometry=d.wkt_ham.map(wkt.loads), crs=4326)
ok = gpd.GeoDataFrame(d[["cati_id", "kayma_m", "kayma_dogu_m", "kayma_kuzey_m", "oturtma"]],
                      geometry=[LineString([a.centroid, b.centroid]) for a, b in zip(ham.geometry, hizali.geometry)], crs=4326)
ok = ok[ok.kayma_m >= 0.5]
sinir = gpd.read_file("sinirlar.geojson")
x1, y1, x2, y2 = hizali.total_bounds
m = folium.Map(location=[(y1 + y2) / 2, (x1 + x2) / 2], zoom_start=13, max_zoom=21, tiles=None)
folium.TileLayer("https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
                 attr="Esri World Imagery", name="Uydu (Esri)", max_zoom=21, max_native_zoom=19).add_to(m)
folium.TileLayer("OpenStreetMap", name="Harita (OSM)", show=False).add_to(m)   # varsayilan altlik uydu
SR = {"ataturk_osb": ("Atatürk OSB sınırı", "#ff3b30"), "cigli": ("Çiğli ilçe sınırı", "#ff9f0a"), "ikcu_kampus": ("İKÇÜ kampüsü", "#bf5af2")}
for e, (ad, renk) in SR.items():
    folium.GeoJson(sinir[sinir.etiket == e], name=ad, style_function=lambda f, c=renk: {"color": c, "weight": 3, "fill": False}).add_to(m)
OTR = {"elle düzeltildi": "#64d2ff", "yüksek": "#34c759", "orta": "#ffd60a", "düşük (yalnız blok)": "#ff3b30"}
folium.GeoJson(ham, name="Ham ayak izi (hizalama öncesi)", show=False,
               style_function=lambda f: {"color": "#d1d1d6", "weight": 1.5, "dashArray": "4 4", "fill": False}).add_to(m)
folium.GeoJson(ok, name="Kayma okları (≥0,5 m)",
               style_function=lambda f: {"color": "#ffffff" if f["properties"]["kayma_m"] < 2 else
                                         "#ffd60a" if f["properties"]["kayma_m"] < 4 else "#ff3b30", "weight": 2.5},
               tooltip=folium.GeoJsonTooltip(fields=["cati_id", "kayma_m", "kayma_dogu_m", "kayma_kuzey_m"],
                                             aliases=["çatı", "kayma m", "doğu m", "kuzey m"])).add_to(m)
for b, ad in (("ataturk_osb", "SAM maskesi — OSB"), ("cigli_kamu", "SAM maskesi — kamu")):
    folium.GeoJson(sam_g[sam_g.bolge == b], name=ad,
                   style_function=lambda f: {"color": "#ff2d55" if f["properties"]["supheli"] == "EVET" else "#0a84ff",
                                             "fillColor": "#ff2d55" if f["properties"]["supheli"] == "EVET" else "#0a84ff",
                                             "weight": 1, "fillOpacity": 0.45},
                   tooltip=folium.GeoJsonTooltip(fields=alanlar, aliases=etiket)).add_to(m)
folium.GeoJson(hizali, name="Hizalı ayak izi (oturtma güveni rengi)",
               style_function=lambda f: {"color": OTR[f["properties"]["oturtma"]], "weight": 2, "fill": False},
               tooltip=folium.GeoJsonTooltip(fields=alanlar, aliases=etiket)).add_to(m)
folium.LayerControl(collapsed=False).add_to(m)
m.get_root().html.add_child(folium.Element("""
<div style="position:fixed;bottom:24px;left:12px;z-index:9999;background:#fff;padding:8px 12px;border-radius:6px;
font:12px sans-serif;line-height:1.7;box-shadow:0 1px 4px rgba(0,0,0,.3)">
<b>Oturtma güveni</b> (hizalı ayak izi çizgisi)<br>
<span style="color:#34c759">■</span> yüksek: kazanç ≥ 1,5 &nbsp; <span style="color:#d4a800">■</span> orta: 1,15–1,5<br>
<span style="color:#ff3b30">■</span> düşük: yalnız blok kayması &nbsp; <span style="color:#64d2ff">■</span> elle düzeltildi<br>
<b>SAM</b> <span style="color:#0a84ff">■</span> maske &nbsp; <span style="color:#ff2d55">■</span> şüpheli maske<br>
<b>Ham ayak izi</b> gri kesik · <b>Kayma oku</b> beyaz &lt;2 m, sarı 2–4, kırmızı &gt;4</div>"""))
m.save("sam_haritasi.html")
sam_g.to_file("sam_poligonlari.geojson", driver="GeoJSON")
print(d.groupby(["bolge", "oturtma"]).agg(cati=("cati_id", "size"), kayma_ort_m=("kayma_m", "mean"),
      kaplama_ort=("kaplama", "mean"), supheli=("supheli", lambda x: (x == "EVET").sum())).round(2))
files.download("sam_haritasi.html")
m

Output hidden; open in https://colab.research.google.com to view.

In [9]:
# 8) SAM poligonlarını indir (ayrı hücre: Chrome aynı anda iki indirmeyi engelliyor)
from google.colab import files
import os; os.chdir("/content/veri")
files.download("sam_poligonlari.geojson")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---
## Multimask karşılaştırması (2026-10-22)
Soru: beşik / kırma çatıda SAM tek yüzü mü seçiyor, maskeleri birleştirmek düzeltir mi? **Sıra: 1 → 2 → 3 → 9 → 10 → 11** (4–8 gerekmez; yaptıysan da sorun yok).
Her çatıda SAM'in 3 aday maskesi ayrı ölçülür ve 3 yöntem karşılaştırılır:
- **A — mevcut**: ayak izine en çok benzeyen tek maske (şimdiki yöntem)
- **U — hepsinin birleşimi**: 3 maskenin birleşimi
- **S — seçici birleşim**: yalnız ayak izinin dışına %30'dan az taşan maskelerin birleşimi (yol / komşu bina girmesin); hiçbiri uymazsa A

Kapsam (`KAPSAM`): şüpheliler + 150 rastgele temiz çatı (kontrol: birleşim iyi sonuçları bozuyor mu?) ~10 dk; `"hepsi"` 1.085 çatı ~40 dk.
Çıktı: `sam_multimask.zip` (tablo + S maskesi kırpılmış ve **kırpılmamış** hali → taşma ilk kez görünür).

In [10]:
# 9) Multimask koşusu (3. hücre: model yüklü olmalı)
import os, ast, time, numpy as np, pandas as pd
from PIL import Image
os.chdir("/content/veri")
KAPSAM = "supheli+kontrol"   # ya da "hepsi"
TASMA_ESIK = 0.3
df = pd.read_csv("sam_girdi.csv")
if os.path.exists("sam_sonuc_tam.csv"):
    onc = pd.read_csv("sam_sonuc_tam.csv")
else:   # 4. hücre çalışmadıysa önceki sonuçlardan
    onc = df.merge(pd.read_csv("onceki/sam_sonuc.csv"), on="cati_id")
    dk = onc.dislama_nedeni.fillna("").astype(str)
    onc["supheli"] = ((onc.kaplama < 0.7) | (onc.tasma > 0.3) | (onc.guven < 0.85) | (onc.bolum == "supheli")
                      | dk.str.contains("gorsel_sorun"))
kontrol = set(onc[~onc.supheli].sample(150, random_state=0).cati_id)
secim = df if KAPSAM == "hepsi" else df[df.cati_id.isin(set(onc.cati_id[onc.supheli]) | kontrol)]
grup = onc.set_index("cati_id").supheli.map({True: "supheli", False: "temiz"})
CIKTI = "sam_multimask.csv"
os.makedirs("sam_maskeler_mm/kirpilmis", exist_ok=True); os.makedirs("sam_maskeler_mm/ham", exist_ok=True)
bitmis = set(pd.read_csv(CIKTI).cati_id) if os.path.exists(CIKTI) else set()
kalan = secim[~secim.cati_id.isin(bitmis)]
print(f"{len(secim)} çatı ({KAPSAM}), {len(bitmis)} bitmiş, {len(kalan)} kalan")

def olc(ham, ayak, px2):
    m = ham & ayak
    return dict(alan=round(m.sum() * px2, 1), kaplama=round(m.sum() / max(ayak.sum(), 1), 3),
                tasma=round((ham & ~ayak).sum() / max(ham.sum(), 1), 3),
                iou=round((ham & ayak).sum() / max((ham | ayak).sum(), 1), 3))
sonuc, t0 = [], time.time()
def yaz():
    global sonuc
    if sonuc:
        pd.DataFrame(sonuc).to_csv(CIKTI, mode="a", header=not os.path.exists(CIKTI), index=False); sonuc = []
for k, r in enumerate(kalan.itertuples(), 1):
    img = np.array(Image.open(r.goruntu).convert("RGB"))
    kutu = np.array(ast.literal_eval(r.kutu_px), dtype=float)
    ayak = np.array(Image.open(r.maske)) > 0
    px2 = r.yer_m_px ** 2
    predictor.set_image(img)
    masks, scores, _ = predictor.predict(box=kutu, multimask_output=True)
    iou = [(mm & ayak).sum() / max((mm | ayak).sum(), 1) for mm in masks]
    i = int(np.argmax(np.array(iou) + 1e-3 * scores))
    satir = dict(cati_id=r.cati_id, grup=grup.get(r.cati_id, "temiz"), secilen=i)
    for j, mm in enumerate(masks):
        o = olc(mm, ayak, px2)
        satir.update({f"m{j}_kaplama": o["kaplama"], f"m{j}_tasma": o["tasma"], f"m{j}_skor": round(float(scores[j]), 3)})
    uygun = [mm for mm in masks if (mm & ~ayak).sum() / max(mm.sum(), 1) <= TASMA_ESIK]
    yontem = {"A": masks[i], "U": masks.any(axis=0), "S": np.logical_or.reduce(uygun) if uygun else masks[i]}
    for ad, ham in yontem.items():
        satir.update({f"{ad}_{a}": v for a, v in olc(ham, ayak, px2).items()})
    s_ham = yontem["S"]
    Image.fromarray(((s_ham & ayak) * 255).astype(np.uint8)).save(f"sam_maskeler_mm/kirpilmis/c{r.cati_id:04d}.png")
    Image.fromarray((s_ham * 255).astype(np.uint8)).save(f"sam_maskeler_mm/ham/c{r.cati_id:04d}.png")
    sonuc.append(satir)
    if k % 25 == 0:
        yaz(); h = (time.time() - t0) / k
        print(f"{k}/{len(kalan)}  ~{h:.1f} sn/çatı, kalan ~{h*(len(kalan)-k)/60:.0f} dk")
yaz()
mm = pd.read_csv(CIKTI).drop_duplicates("cati_id", keep="last")
mm = mm[mm.cati_id.isin(secim.cati_id)]
assert set(mm.cati_id) == set(secim.cati_id), "eksik çatı var"
print("bitti:", len(mm), mm.grup.value_counts().to_dict())

278 çatı (supheli+kontrol), 0 bitmiş, 278 kalan
25/278  ~2.3 sn/çatı, kalan ~10 dk
50/278  ~2.2 sn/çatı, kalan ~8 dk
75/278  ~2.2 sn/çatı, kalan ~7 dk
100/278  ~2.2 sn/çatı, kalan ~7 dk
125/278  ~2.2 sn/çatı, kalan ~6 dk
150/278  ~2.2 sn/çatı, kalan ~5 dk
175/278  ~2.2 sn/çatı, kalan ~4 dk
200/278  ~2.2 sn/çatı, kalan ~3 dk
225/278  ~2.2 sn/çatı, kalan ~2 dk
250/278  ~2.2 sn/çatı, kalan ~1 dk
275/278  ~2.2 sn/çatı, kalan ~0 dk
bitti: 278 {'temiz': 150, 'supheli': 128}


In [11]:
# 10) Karşılaştırma: A (mevcut) / U (hepsi) / S (seçici birleşim)
import os, ast, numpy as np, pandas as pd, matplotlib.pyplot as plt
from PIL import Image
os.chdir("/content/veri")
mm = pd.read_csv("sam_multimask.csv").drop_duplicates("cati_id", keep="last").reset_index(drop=True)
def iyi(p): return (mm[f"{p}_kaplama"] >= 0.7) & (mm[f"{p}_tasma"] <= 0.3)
ozet = []
for g, gg in mm.groupby("grup"):
    for p in "AUS":
        ozet.append(dict(grup=g, yontem=p, cati=len(gg), kaplama_medyan=gg[f"{p}_kaplama"].median(),
                         tasma_medyan=gg[f"{p}_tasma"].median(), iou_medyan=gg[f"{p}_iou"].median(),
                         kural_gecen=int(iyi(p)[gg.index].sum()), alan_m2=round(gg[f"{p}_alan"].sum())))
print(pd.DataFrame(ozet).round(3).to_string(index=False))
sup = mm[(mm.grup == "supheli") & (mm.A_kaplama < 0.7)]
print(f"kaplama < 0,7 olan şüpheli {len(sup)} çatı: S ile kurala uyan {int(iyi('S')[sup.index].sum())}, U ile {int(iyi('U')[sup.index].sum())}")
tem = mm[mm.grup == "temiz"]
print(f"temiz {len(tem)} çatı: kurala uymaz hale gelen S ile {int((~iyi('S')[tem.index]).sum())}, U ile {int((~iyi('U')[tem.index]).sum())}")
# görsel: S'nin kaplamayı en çok artırdığı 15 çatı — sol A (mevcut), sağ S (mavi: ayak izi içi, kırmızı: taşma)
tablo = pd.read_csv("sam_girdi.csv").set_index("cati_id")
goster = mm.assign(fark=mm.S_kaplama - mm.A_kaplama).nlargest(15, "fark")
fig, ax = plt.subplots(len(goster), 2, figsize=(7, 3.3 * len(goster)), squeeze=False)
for (a1, a2), q in zip(ax, goster.itertuples()):
    r = tablo.loc[q.cati_id]
    img = np.array(Image.open(r.goruntu).convert("RGB")); ayak = np.array(Image.open(r.maske)) > 0
    s_ham = np.array(Image.open(f"sam_maskeler_mm/ham/c{q.cati_id:04d}.png")) > 0
    yol = f"sam_maskeler/c{q.cati_id:04d}.png"
    ma = np.array(Image.open(yol if os.path.exists(yol) else "onceki/" + yol)) > 0
    p = np.array(ast.literal_eval(r.poligon_px))
    for a, baslik in ((a1, f"c{q.cati_id:04d} A kap={q.A_kaplama}"), (a2, f"S kap={q.S_kaplama} taş={q.S_tasma}")):
        a.imshow(img); a.plot(p[:, 0], p[:, 1], c="yellow", lw=1); a.set_title(baslik, fontsize=8); a.axis("off")
    a1.imshow(np.ma.masked_where(~ma, ma), alpha=0.45, cmap="winter")
    a2.imshow(np.ma.masked_where(~(s_ham & ayak), s_ham), alpha=0.45, cmap="winter")
    a2.imshow(np.ma.masked_where(~(s_ham & ~ayak), s_ham), alpha=0.6, cmap="autumn")
plt.tight_layout(); plt.savefig("sam_multimask_onizleme.jpg", dpi=90); plt.show()

Output hidden; open in https://colab.research.google.com to view.

In [12]:
# 11) Multimask sonuçlarını indir
import os, zipfile
from google.colab import files
os.chdir("/content/veri")
with zipfile.ZipFile("/content/sam_multimask.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in ["sam_multimask.csv", "sam_multimask_onizleme.jpg"]:
        if os.path.exists(f): z.write(f)
    for kok, _, dosyalar in os.walk("sam_maskeler_mm"):
        for f in sorted(dosyalar): z.write(os.path.join(kok, f))
files.download("/content/sam_multimask.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [13]:
# 12) SAM AYARI ICIN ADAY MASKELER (2026-10-24) - tek hucre: kurulum + Drive zip + model + 178 cati
# k0 = mevcut kutu | k10 = kutu %10 genis | k0n = kutu + merkez noktasi; her istemde 3 aday KIRPILMADAN kaydedilir
import os, glob, zipfile, subprocess, shutil, ast, time
from google.colab import files
# veri: sam_ayar_girdi_1/2.zip (sam_girdi.csv + 178 goruntu) dosya panelinden /content altina yuklenir (Drive izni gerekmez)
if not os.path.exists("/content/sam_vit_h_4b8939.pth"):
    subprocess.run("pip -q install git+https://github.com/facebookresearch/segment-anything.git", shell=True, check=True)
    subprocess.run("wget -q -c -P /content https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth", shell=True, check=True)
if not os.path.exists("/content/veri/sam_girdi.csv"):
    for p in ("/content/sam_ayar_girdi_1.zip", "/content/sam_ayar_girdi_2.zip"):
        zipfile.ZipFile(p).extractall("/content/veri")
import numpy as np, pandas as pd, torch
from PIL import Image
from segment_anything import sam_model_registry, SamPredictor
predictor = SamPredictor(sam_model_registry["vit_h"](checkpoint="/content/sam_vit_h_4b8939.pth").to("cuda"))
print("model hazir:", torch.cuda.get_device_name(0))
os.chdir("/content/veri")
IDS = [4, 6, 7, 8, 9, 10, 15, 17, 25, 32, 33, 35, 36, 37, 45, 48, 56, 64, 75, 76, 79, 80, 90, 100, 101, 107, 113, 114,
       118, 125, 139, 140, 141, 144, 150, 157, 158, 159, 160, 171, 176, 190, 191, 224, 236, 240, 248, 253, 286, 316, 319,
       326, 329, 338, 360, 381, 392, 408, 412, 418, 427, 432, 434, 437, 441, 442, 447, 453, 457, 459, 465, 466, 467, 468,
       483, 485, 487, 505, 508, 514, 519, 528, 533, 557, 560, 568, 587, 595, 598, 602, 611, 625, 643, 647, 654, 657, 684,
       700, 711, 712, 727, 743, 755, 791, 828, 833, 857, 897, 919, 949, 986, 997, 998, 1001, 1004, 1055, 1056, 1057, 1063,
       1068, 1074, 1076, 1080, 1100, 1106, 1124, 1132, 1141, 1158, 1163, 1164, 1186, 1205, 1225, 1226, 1229, 1254, 1280,
       1293, 1307, 1317, 1324, 1329, 1334, 1341, 1358, 1395, 1397, 1398, 1399, 1400, 1402, 1404, 1414, 1415, 1417, 1423,
       1427, 1438, 1440, 1443, 1444, 1445, 1446, 1448, 1449, 1450, 1455, 1457, 1458, 1459, 1460, 1463, 1468, 1470, 1476,
       1488, 1495]
df = pd.read_csv("sam_girdi.csv").set_index("cati_id")
assert set(IDS) <= set(df.index), "sam_girdi.csv'de olmayan cati var"
os.makedirs("sam_aday", exist_ok=True)
t0 = time.time()
for k, cid in enumerate(IDS, 1):
    yol = f"sam_aday/c{cid:04d}.npz"
    if os.path.exists(yol):
        continue
    r = df.loc[cid]
    img = np.array(Image.open(r.goruntu).convert("RGB"))
    H, W = img.shape[:2]
    kutu = np.array(ast.literal_eval(r.kutu_px), dtype=float)
    w, h = kutu[2] - kutu[0], kutu[3] - kutu[1]
    k10 = np.clip(kutu + np.array([-.1 * w, -.1 * h, .1 * w, .1 * h]), 0, [W - 1, H - 1, W - 1, H - 1])
    merkez = np.array([[(kutu[0] + kutu[2]) / 2, (kutu[1] + kutu[3]) / 2]])
    predictor.set_image(img)
    out = {"boyut": np.array([H, W])}
    for ad, arg in (("k0", dict(box=kutu)), ("k10", dict(box=k10)),
                    ("k0n", dict(box=kutu, point_coords=merkez, point_labels=np.array([1])))):
        masks, scores, _ = predictor.predict(multimask_output=True, **arg)
        out[ad + "_m"] = np.packbits(masks, axis=-1)
        out[ad + "_s"] = scores
    np.savez_compressed(yol, **out)
    if k % 25 == 0:
        print(f"{k}/{len(IDS)}  ~{(time.time() - t0) / k:.1f} sn/cati")
assert len(os.listdir("sam_aday")) == len(IDS), "eksik cati var, hucreyi tekrar calistir"
with zipfile.ZipFile("/content/sam_aday.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted(os.listdir("sam_aday")):
        z.write(os.path.join("sam_aday", f))
print("bitti:", len(IDS), "cati, zip %.1f MB" % (os.path.getsize("/content/sam_aday.zip") / 1e6))
files.download("/content/sam_aday.zip")

model hazir: Tesla T4
25/178  ~2.3 sn/cati
50/178  ~2.2 sn/cati
75/178  ~2.2 sn/cati
100/178  ~2.2 sn/cati
125/178  ~2.2 sn/cati
150/178  ~2.2 sn/cati
175/178  ~2.2 sn/cati
bitti: 178 cati, zip 1.3 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>